In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from cmdstanpy import CmdStanModel

import arviz as az

C:\Users\Josh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the 2026 dataset: https://memory.psych.upenn.edu/Data_Archive#2026

In [8]:
# Get data
data = pd.read_csv('data/Exp3_AllData.csv', header=0, sep=',')
data.head()
data = data[~np.isnan(data["Output_Rank"])]

# Convert to ints
data["Listnum"] = data["Listnum"].astype(int)
data["Spatial_Input_Order"] = data["Spatial_Input_Order"].astype(int)
data["Temporal_Input_Position"] = data["Temporal_Input_Position"].astype(int)
data["Spatial_Recall_Order"] = data["Spatial_Recall_Order"].astype(int)
data["Length"] = data["Length"].astype(int)
data["Correct"] = data["Correct"].astype(int)
data["Distance_From_Correct"] = data["Distance_From_Correct"].astype(int)
data["Serial_Pos_Encoded"] = data["Serial_Pos_Encoded"].astype(int)
data["Output_Rank"] = data["Output_Rank"].astype(int)

# Filter for only spatial and varied data
data = data[data["Order_Type"] == 'Spatial']
data = data[data["List_Type"] == 'Varied']

# Only use a fraction of the data
train_percent = 0.005
train_n = int(data.shape[0]*train_percent)
data = data.sample(n=train_n)

data_dict = {
    # Sizes
    "N": data.shape[0],
    "S": data["Uniqueid"].nunique(),
    # IDs
    "subject_index": data["Uniqueid"].astype("category").cat.codes + 1,
    # Experiment
    #"list_number": data.Listnum.values,
    "spatial_input_order": data.Spatial_Input_Order.values,
    #"temporal_input_pos": data.Temporal_Input_Position.values,
    "list_length": data.Length.values,
    # Results
    "spatial_recall_order": data.Spatial_Recall_Order.values,
    "correct": data.Correct.values,
    "distance_from_correct": data.Distance_From_Correct.values,
    #"output_rank": data.Output_Rank.values,
    "rt": data.Initial_RT_Time.values,
}

In [6]:
# Compile model
model = CmdStanModel(stan_file="stan/model.stan")

18:21:47 - cmdstanpy - INFO - compiling stan file C:\Users\Josh\Box\Documents\COGS-4210_Cognitive_Modeling\Repo\project\stan\model.stan to exe file C:\Users\Josh\Box\Documents\COGS-4210_Cognitive_Modeling\Repo\project\stan\model.exe
18:22:05 - cmdstanpy - INFO - compiled model executable: C:\Users\Josh\Box\Documents\COGS-4210_Cognitive_Modeling\Repo\project\stan\model.exe


In [9]:
# Sample model
fit = model.sample(data=data_dict, chains=4, iter_sampling=2500, iter_warmup=1000)

# Display sampling diagnostics
print(fit.diagnose())

18:24:04 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]





chain 1:   3%|▎         | 100/3500 [00:01<01:05, 52.11it/s, (Warmup)]


chain 1:   6%|▌         | 200/3500 [00:04<01:21, 40.65it/s, (Warmup)]


chain 1:   9%|▊         | 300/3500 [00:08<01:30, 35.44it/s, (Warmup)]


chain 1:  11%|█▏        | 400/3500 [00:11<01:34, 32.75it/s, (Warmup)]


chain 1:  14%|█▍        | 500/3500 [00:14<01:34, 31.73it/s, (Warmup)]


chain 1:  17%|█▋        | 600/3500 [00:18<01:33, 30.97it/s, (Warmup)]


chain 1:  20%|██        | 700/3500 [00:21<01:31, 30.59it/s, (Warmup)]


chain 1:  23%|██▎       | 800/3500 [00:24<01:29, 30.25it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:28<01:27, 29.84it/s, (Warmup)]





chain 1:  29%|██▊       | 1000/3500 [00:31<01:23, 29.77it/s, (Sampling)]


chain 1:  31%|███▏      | 1100/3500 [00:35<01:22, 29.18it/s, (Sampling)]


chain 1:  34%|███▍      | 1200/3500 [00:38<01:18, 29.13it/s, (Sampling)]


chai


18:26:07 - cmdstanpy - INFO - CmdStan done processing.
18:26:07 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'model.stan', line 54, column 8 to column 58)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'model.stan', line 54, column 8 to column 58)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'model.stan', line 54, column 8 to column 58)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'model.stan', line 54, column 8 to column 58)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'model.stan', line 54, column 8 to column 58)
Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'model.stan', line 54, column 8 to column 58)
	Exception: lognormal_lpdf: Scale parameter is inf, but must be positive finite! (in 'model.st


Checking sampler transitions treedepth.
9959 of 10000 (99.59%) transitions hit the maximum treedepth limit of 10, or 2^10 leapfrog steps.
Trajectories that are prematurely terminated due to this limit will result in slow exploration.
For optimal performance, increase this limit.

Checking sampler transitions for divergences.
33 of 10000 (0.33%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

The following parameters had fewer than 0.001 effective draws per transition:
  sigma_recall[7], recall[2], recall[3], recall[6], recall[9], recall[10], recall[14], recall[17], recall[18], recall[19], recall[23], recall[26], recall[27], recall[32], recall[34], recall[37], recall[40], recall[41], recall[43], recall[